# 03 - Construção da Camada Gold

Nesta etapa, os dados tratados e harmonizados da Camada Silver serão utilizados para a construção da Camada Gold.

A Gold será responsável por organizar os dados para consumo analítico, estruturando tabelas fato, dimensões e indicadores derivados da PNAD Contínua, com foco na análise dos entregadores por aplicativo em 2022 e 2024.

Antes da criação das tabelas analíticas, será definida a granularidade do modelo e validada a integridade dos dados provenientes da Camada Silver.

In [0]:
# Leitura da tabela Silver consolidada

df_silver = spark.table("silver_entregadores_pnad")

print("Total de registros na Silver:")
print(df_silver.count())

print("\nSchema da Silver:")
df_silver.printSchema()

## 1. Definição da granularidade do modelo analítico

A tabela fato será construída a partir da base harmonizada da Camada Silver, preservando a granularidade dos microdados da PNAD Contínua.

Cada registro da tabela fato corresponde a uma observação individual presente nos microdados em determinado período da pesquisa. Dessa forma, um registro da tabela fato não deve ser interpretado diretamente como uma pessoa da população brasileira.

Por se tratar de uma pesquisa amostral, as estimativas populacionais serão produzidas mediante a utilização do peso amostral associado a cada observação.

A manutenção dos registros da base harmonizada, em lugar da seleção exclusiva dos entregadores por aplicativo, permite preservar diferentes possibilidades de comparação entre grupos de trabalhadores nas etapas posteriores da análise.

A população de interesse do projeto será identificada por meio do indicador `entregador_plataformizado`, construído na Camada Silver.

### Grão da tabela fato

**Uma linha = uma observação individual da PNAD Contínua em determinado período da pesquisa.**

Essa definição orientará a construção das dimensões, das medidas e dos indicadores que compõem a Camada Gold.

In [0]:
# Validação inicial do grão da futura tabela fato

from pyspark.sql.functions import count, sum as spark_sum, col

display(
    df_silver
    .groupBy("ano_pnad")
    .agg(
        count("*").alias("registros_silver"),
        spark_sum(
            col("entregador_plataformizado")
        ).alias("registros_entregadores_plataformizados")
    )
    .orderBy("ano_pnad")
)

## 2. Modelagem dimensional da Camada Gold

A Camada Gold será estruturada por meio de um modelo dimensional voltado ao consumo analítico dos dados harmonizados da PNAD Contínua.

O modelo adota uma tabela fato central, denominada `fact_trabalhador`, relacionada a dimensões que representam características temporais, sociodemográficas e ocupacionais presentes na base.

### Tabela fato

- `fact_trabalhador`: preserva o grão individual dos registros da PNAD e concentra as medidas e os indicadores necessários às análises, incluindo o peso amostral e os indicadores relacionados ao trabalho por plataformas.

### Dimensões

- `dim_tempo`: identifica o período da pesquisa;
- `dim_sexo`: organiza as categorias de sexo;
- `dim_cor_raca`: organiza as categorias de cor ou raça;
- `dim_faixa_etaria`: organiza os trabalhadores segundo grupos de idade;
- `dim_posicao_ocupacao`: representa a posição do trabalhador na ocupação principal;
- `dim_atividade_principal`: representa a atividade econômica principal registrada na PNAD.

A definição das dimensões procura responder às necessidades analíticas do projeto sem reproduzir integralmente a estrutura dos microdados na Camada Gold.

O peso amostral será mantido na tabela fato como elemento necessário à produção das estimativas populacionais. Os registros amostrais e as estimativas ponderadas serão tratados como medidas distintas ao longo das análises.

### 2.1. Dimensão Tempo

A dimensão `dim_tempo` organiza os períodos da PNAD Contínua utilizados no projeto.

Embora os dados sejam identificados pelo ano de referência, os dois levantamentos correspondem a períodos específicos da pesquisa. A dimensão temporal permite centralizar essa informação e evita que a identificação do período seja reproduzida diretamente em diferentes estruturas analíticas.

Cada registro da dimensão corresponde a um período da PNAD incorporado ao projeto.

In [0]:
# Construção da dimensão tempo

from pyspark.sql.functions import col, when

dim_tempo = (
    df_silver
    .select("ano_pnad")
    .distinct()
    .withColumn(
        "id_tempo",
        when(col("ano_pnad") == 2022, 1)
        .when(col("ano_pnad") == 2024, 2)
    )
    .withColumn(
        "periodo_pnad",
        when(col("ano_pnad") == 2022, "4º trimestre de 2022")
        .when(col("ano_pnad") == 2024, "3º trimestre de 2024")
    )
    .select(
        "id_tempo",
        "ano_pnad",
        "periodo_pnad"
    )
    .orderBy("id_tempo")
)

display(dim_tempo)

In [0]:
# Persistência da dimensão tempo na Camada Gold

tabela_dim_tempo = "gold_dim_tempo"

(
    dim_tempo
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_dim_tempo)
)

print("Dimensão criada com sucesso:")
print(tabela_dim_tempo)

In [0]:
# Validação da dimensão tempo persistida

df_teste_dim_tempo = spark.table("gold_dim_tempo")

print("Total de registros:")
print(df_teste_dim_tempo.count())

display(
    df_teste_dim_tempo
    .orderBy("id_tempo")
)

### 2.2. Dimensão Sexo

A dimensão `dim_sexo` organiza as categorias de sexo presentes nos microdados da PNAD Contínua.

A dimensão utiliza a variável harmonizada construída na Camada Silver, permitindo que a característica sociodemográfica seja relacionada aos registros da tabela fato por meio de uma chave dimensional.

In [0]:
# Construção da dimensão sexo

from pyspark.sql.functions import col, when

dim_sexo = (
    df_silver
    .select("sexo", "sexo_desc")
    .distinct()
    .withColumn(
        "id_sexo",
        when(col("sexo") == "1", 1)
        .when(col("sexo") == "2", 2)
        .otherwise(9)
    )
    .select(
        "id_sexo",
        col("sexo").alias("codigo_sexo_pnad"),
        "sexo_desc"
    )
    .orderBy("id_sexo")
)

display(dim_sexo)

In [0]:
# Persistência da dimensão sexo na Camada Gold

tabela_dim_sexo = "gold_dim_sexo"

(
    dim_sexo
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_dim_sexo)
)

print("Dimensão criada com sucesso:")
print(tabela_dim_sexo)

### 2.3. Dimensão Cor ou Raça

A dimensão `dim_cor_raca` organiza as categorias de cor ou raça presentes nos microdados da PNAD Contínua.

A dimensão utiliza a variável harmonizada construída na Camada Silver, preservando a codificação original da PNAD e sua descrição correspondente para posterior relacionamento com a tabela fato.

In [0]:
# Construção da dimensão cor ou raça

from pyspark.sql.functions import col, when

dim_cor_raca = (
    df_silver
    .select("cor_raca", "cor_raca_desc")
    .distinct()
    .withColumn(
        "id_cor_raca",
        when(col("cor_raca") == "1", 1)
        .when(col("cor_raca") == "2", 2)
        .when(col("cor_raca") == "3", 3)
        .when(col("cor_raca") == "4", 4)
        .when(col("cor_raca") == "5", 5)
        .when(col("cor_raca") == "9", 9)
        .otherwise(99)
    )
    .select(
        "id_cor_raca",
        col("cor_raca").alias("codigo_cor_raca_pnad"),
        "cor_raca_desc"
    )
    .orderBy("id_cor_raca")
)

display(dim_cor_raca)

In [0]:
# Persistência da dimensão cor ou raça na Camada Gold

tabela_dim_cor_raca = "gold_dim_cor_raca"

(
    dim_cor_raca
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_dim_cor_raca)
)

print("Dimensão criada com sucesso:")
print(tabela_dim_cor_raca)

### 2.4. Dimensão Faixa Etária

A dimensão `dim_faixa_etaria` organiza as idades registradas na PNAD Contínua em grupos etários utilizados nas análises do projeto.

As faixas são derivadas da variável `idade_anos`, construída na Camada Silver, permitindo comparar a composição etária dos trabalhadores entre os períodos analisados sem eliminar a informação de idade original disponível na base harmonizada.

Foram consideradas as seguintes faixas: 14 a 17 anos, 18 a 24 anos, 25 a 34 anos, 35 a 44 anos, 45 a 54 anos, 55 a 64 anos e 65 anos ou mais.

In [0]:
# Construção inicial das faixas etárias

from pyspark.sql.functions import col, when

df_faixa_etaria = (
    df_silver
    .withColumn(
        "id_faixa_etaria",
        when(col("idade_anos").between(14, 17), 1)
        .when(col("idade_anos").between(18, 24), 2)
        .when(col("idade_anos").between(25, 34), 3)
        .when(col("idade_anos").between(35, 44), 4)
        .when(col("idade_anos").between(45, 54), 5)
        .when(col("idade_anos").between(55, 64), 6)
        .when(col("idade_anos") >= 65, 7)
        .otherwise(99)
    )
    .withColumn(
        "faixa_etaria",
        when(col("id_faixa_etaria") == 1, "14 a 17 anos")
        .when(col("id_faixa_etaria") == 2, "18 a 24 anos")
        .when(col("id_faixa_etaria") == 3, "25 a 34 anos")
        .when(col("id_faixa_etaria") == 4, "35 a 44 anos")
        .when(col("id_faixa_etaria") == 5, "45 a 54 anos")
        .when(col("id_faixa_etaria") == 6, "55 a 64 anos")
        .when(col("id_faixa_etaria") == 7, "65 anos ou mais")
        .otherwise("Fora do universo analítico")
    )
)

display(
    df_faixa_etaria
    .groupBy("id_faixa_etaria", "faixa_etaria")
    .count()
    .orderBy("id_faixa_etaria")
)

In [0]:
# Construção da dimensão faixa etária

dim_faixa_etaria = (
    df_faixa_etaria
    .select(
        "id_faixa_etaria",
        "faixa_etaria"
    )
    .distinct()
    .orderBy("id_faixa_etaria")
)

display(dim_faixa_etaria)

In [0]:
# Persistência da dimensão faixa etária na Camada Gold

tabela_dim_faixa_etaria = "gold_dim_faixa_etaria"

(
    dim_faixa_etaria
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_dim_faixa_etaria)
)

print("Dimensão criada com sucesso:")
print(tabela_dim_faixa_etaria)

### 2.5. Dimensão Posição na Ocupação

A dimensão `dim_posicao_ocupacao` organiza a posição do trabalhador na ocupação principal, conforme classificação presente na PNAD Contínua.

A construção da dimensão parte da variável `V4012`, preservada na Camada Silver. Antes da definição das categorias analíticas, são inspecionados os valores efetivamente presentes na base, incluindo eventuais registros não aplicáveis.

In [0]:
# Inspeção dos valores de V4012 presentes na Silver

display(
    df_silver
    .groupBy("V4012")
    .count()
    .orderBy("V4012")
)

In [0]:
# Construção da dimensão posição na ocupação

from pyspark.sql.functions import col, when

dim_posicao_ocupacao = (
    df_silver
    .select("V4012")
    .distinct()
    .withColumn(
        "id_posicao_ocupacao",
        when(col("V4012") == "1", 1)
        .when(col("V4012") == "2", 2)
        .when(col("V4012") == "3", 3)
        .when(col("V4012") == "4", 4)
        .when(col("V4012") == "5", 5)
        .when(col("V4012") == "6", 6)
        .when(col("V4012") == "7", 7)
        .otherwise(99)
    )
    .withColumn(
        "posicao_ocupacao",
        when(col("V4012") == "1", "Trabalhador doméstico")
        .when(col("V4012") == "2", "Militar")
        .when(col("V4012") == "3", "Empregado do setor privado")
        .when(col("V4012") == "4", "Empregado do setor público")
        .when(col("V4012") == "5", "Empregador")
        .when(col("V4012") == "6", "Conta própria")
        .when(col("V4012") == "7", "Trabalhador familiar auxiliar")
        .otherwise("Não aplicável")
    )
    .select(
        "id_posicao_ocupacao",
        col("V4012").alias("codigo_posicao_pnad"),
        "posicao_ocupacao"
    )
    .orderBy("id_posicao_ocupacao")
)

display(dim_posicao_ocupacao)

In [0]:
# Persistência da dimensão posição na ocupação na Camada Gold

tabela_dim_posicao_ocupacao = "gold_dim_posicao_ocupacao"

(
    dim_posicao_ocupacao
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_dim_posicao_ocupacao)
)

print("Dimensão criada com sucesso:")
print(tabela_dim_posicao_ocupacao)

### 2.6. Dimensão Atividade Principal

A dimensão `dim_atividade_principal` será construída a partir da variável `V4013`, que identifica a atividade principal registrada nos microdados da PNAD Contínua.

Como essa variável possui maior diversidade de códigos, sua estrutura será inicialmente inspecionada antes da definição da forma final da dimensão.

Essa etapa permite avaliar se a atividade principal será preservada no nível do código original ou se será necessário construir agrupamentos analíticos mais adequados aos objetivos do projeto.

In [0]:
# Inspeção da variável V4013

from pyspark.sql.functions import count

print("Quantidade de códigos distintos em V4013:")
print(
    df_silver
    .select("V4013")
    .distinct()
    .count()
)

print("\nCódigos mais frequentes:")

display(
    df_silver
    .groupBy("V4013")
    .agg(
        count("*").alias("registros")
    )
    .orderBy(col("registros").desc())
    .limit(30)
)

A inspeção da variável `V4013` identificou elevada diversidade de códigos de atividade principal.

Diante dessa estrutura, optou-se por preservar, nesta versão do modelo, o código de atividade registrado na PNAD, sem produzir agrupamentos adicionais que não tenham sido previamente definidos e validados.

Essa escolha mantém a rastreabilidade em relação aos microdados originais e permite que classificações mais detalhadas sejam incorporadas posteriormente ao modelo analítico.

In [0]:
# Construção da dimensão atividade principal

from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# Códigos válidos de atividade
atividades_validas = (
    df_silver
    .filter(col("V4013").isNotNull() & (col("V4013") != ""))
    .select("V4013")
    .distinct()
)

# Criação de chave substituta ordenada pelo código original
janela = Window.orderBy("V4013")

dim_atividade_valida = (
    atividades_validas
    .withColumn(
        "id_atividade_principal",
        row_number().over(janela)
    )
    .withColumn(
        "descricao_atividade",
        col("V4013")
    )
    .select(
        "id_atividade_principal",
        col("V4013").alias("codigo_atividade_pnad"),
        "descricao_atividade"
    )
)

# Schema explícito para a categoria não aplicável
schema_nao_aplicavel = StructType([
    StructField("id_atividade_principal", IntegerType(), False),
    StructField("codigo_atividade_pnad", StringType(), True),
    StructField("descricao_atividade", StringType(), False)
])

# Categoria técnica para registros fora do universo
dim_atividade_nao_aplicavel = spark.createDataFrame(
    [(999, None, "Não aplicável")],
    schema=schema_nao_aplicavel
)

# União das categorias válidas com a categoria não aplicável
dim_atividade_principal = (
    dim_atividade_valida
    .unionByName(dim_atividade_nao_aplicavel)
    .orderBy("id_atividade_principal")
)

print("Total de registros na dimensão:")
print(dim_atividade_principal.count())

display(dim_atividade_principal)

In [0]:
# Persistência da dimensão atividade principal na Camada Gold

tabela_dim_atividade_principal = "gold_dim_atividade_principal"

(
    dim_atividade_principal
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_dim_atividade_principal)
)

print("Dimensão criada com sucesso:")
print(tabela_dim_atividade_principal)

## 3. Construção da Tabela Fato

A tabela `fact_trabalhador` constitui o núcleo do modelo dimensional da Camada Gold.

Cada registro corresponde a uma observação individual presente nos microdados harmonizados da PNAD Contínua em determinado período da pesquisa.

A tabela fato concentra as chaves que permitem o relacionamento com as dimensões temporais, sociodemográficas e ocupacionais, além das medidas e indicadores utilizados nas análises do projeto.

Entre os elementos preservados na tabela fato estão o peso amostral, necessário à produção das estimativas populacionais, e os indicadores relacionados ao trabalho por plataformas construídos na Camada Silver.

A construção da tabela fato deverá preservar integralmente a quantidade de registros da base Silver, de modo que a modelagem dimensional não produza perdas ou duplicações de observações.

In [0]:
# Construção inicial da tabela fato

from pyspark.sql.functions import col, when, monotonically_increasing_id

fact_trabalhador = (
    df_faixa_etaria

    # Chave da dimensão tempo
    .withColumn(
        "id_tempo",
        when(col("ano_pnad") == 2022, 1)
        .when(col("ano_pnad") == 2024, 2)
    )

    # Chave da dimensão sexo
    .withColumn(
        "id_sexo",
        when(col("sexo") == "1", 1)
        .when(col("sexo") == "2", 2)
        .otherwise(9)
    )

    # Chave da dimensão cor ou raça
    .withColumn(
        "id_cor_raca",
        when(col("cor_raca") == "1", 1)
        .when(col("cor_raca") == "2", 2)
        .when(col("cor_raca") == "3", 3)
        .when(col("cor_raca") == "4", 4)
        .when(col("cor_raca") == "5", 5)
        .when(col("cor_raca") == "9", 9)
        .otherwise(99)
    )

    # Chave da dimensão posição na ocupação
    .withColumn(
        "id_posicao_ocupacao",
        when(col("V4012") == "1", 1)
        .when(col("V4012") == "2", 2)
        .when(col("V4012") == "3", 3)
        .when(col("V4012") == "4", 4)
        .when(col("V4012") == "5", 5)
        .when(col("V4012") == "6", 6)
        .when(col("V4012") == "7", 7)
        .otherwise(99)
    )
)

# Associação com a dimensão atividade principal
fact_trabalhador = (
    fact_trabalhador
    .join(
        dim_atividade_principal.select(
            "id_atividade_principal",
            "codigo_atividade_pnad"
        ),
        fact_trabalhador["V4013"] == dim_atividade_principal["codigo_atividade_pnad"],
        "left"
    )
    .withColumn(
        "id_atividade_principal",
        when(
            col("id_atividade_principal").isNull(),
            999
        ).otherwise(col("id_atividade_principal"))
    )
)

# Seleção das colunas da tabela fato
fact_trabalhador = (
    fact_trabalhador
    .select(
        "id_tempo",
        "id_sexo",
        "id_cor_raca",
        "id_faixa_etaria",
        "id_posicao_ocupacao",
        "id_atividade_principal",
        "idade_anos",
        "peso_amostral",
        "SD14001",
        "entregador_app",
        "app_taxi",
        "app_transporte_passageiros",
        "app_servicos",
        "app_entrega",
        "entregador_plataformizado"
    )
    .withColumn(
        "id_registro_fato",
        monotonically_increasing_id()
    )
)

print("Total de registros na futura tabela fato:")
print(fact_trabalhador.count())

display(fact_trabalhador.limit(20))

In [0]:
# Validação de integridade da tabela fato

from pyspark.sql.functions import count, sum as spark_sum, col

display(
    fact_trabalhador
    .groupBy("id_tempo")
    .agg(
        count("*").alias("registros_fato"),
        
        spark_sum(
            col("entregador_plataformizado")
        ).alias("registros_entregadores"),
        
        spark_sum(
            when(
                col("entregador_plataformizado") == 1,
                col("peso_amostral")
            ).otherwise(0)
        ).alias("estimativa_entregadores")
    )
    .orderBy("id_tempo")
)

In [0]:
# Validação das chaves dimensionais da tabela fato

from pyspark.sql.functions import col, sum as spark_sum, when

display(
    fact_trabalhador.select(
        spark_sum(when(col("id_tempo").isNull(), 1).otherwise(0))
            .alias("nulos_id_tempo"),

        spark_sum(when(col("id_sexo").isNull(), 1).otherwise(0))
            .alias("nulos_id_sexo"),

        spark_sum(when(col("id_cor_raca").isNull(), 1).otherwise(0))
            .alias("nulos_id_cor_raca"),

        spark_sum(when(col("id_faixa_etaria").isNull(), 1).otherwise(0))
            .alias("nulos_id_faixa_etaria"),

        spark_sum(when(col("id_posicao_ocupacao").isNull(), 1).otherwise(0))
            .alias("nulos_id_posicao_ocupacao"),

        spark_sum(when(col("id_atividade_principal").isNull(), 1).otherwise(0))
            .alias("nulos_id_atividade_principal")
    )
)

In [0]:
# Persistência da tabela fato na Camada Gold

tabela_fact_trabalhador = "gold_fact_trabalhador"

(
    fact_trabalhador
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_fact_trabalhador)
)

print("Tabela fato criada com sucesso:")
print(tabela_fact_trabalhador)

In [0]:
# Validação final da tabela fato persistida

df_fact_teste = spark.table("gold_fact_trabalhador")

print("Total de registros na tabela fato persistida:")
print(df_fact_teste.count())

print("\nRegistros de entregadores plataformizados:")
print(
    df_fact_teste
    .filter(col("entregador_plataformizado") == 1)
    .count()
)

display(
    df_fact_teste
    .groupBy("id_tempo")
    .agg(
        count("*").alias("registros_fato"),
        spark_sum(
            when(
                col("entregador_plataformizado") == 1,
                col("peso_amostral")
            ).otherwise(0)
        ).alias("estimativa_entregadores")
    )
    .orderBy("id_tempo")
)

## 4. Construção das Tabelas Analíticas

Além do modelo dimensional, a Camada Gold incorpora tabelas agregadas destinadas ao consumo analítico dos dados.

Essas tabelas são produzidas a partir da tabela fato e de suas dimensões, preservando a distinção entre registros amostrais e estimativas populacionais ponderadas.

A construção dos indicadores segue as perguntas analíticas definidas para o projeto e busca disponibilizar estruturas de consulta mais simples para análises, visualizações e desenvolvimento posterior do Observatório de Dados dos Entregadores por Aplicativo.

### 4.1. Indicadores Gerais

A primeira tabela analítica apresenta, para cada período da PNAD considerado no projeto, o número de registros amostrais classificados como entregadores plataformizados e sua correspondente estimativa populacional obtida mediante aplicação do peso amostral.

In [0]:
# Construção dos indicadores gerais de entregadores plataformizados

from pyspark.sql.functions import count, sum as spark_sum, col

gold_indicadores_gerais = (
    spark.table("gold_fact_trabalhador")
    .filter(col("entregador_plataformizado") == 1)
    .join(
        spark.table("gold_dim_tempo"),
        on="id_tempo",
        how="left"
    )
    .groupBy(
        "id_tempo",
        "ano_pnad",
        "periodo_pnad"
    )
    .agg(
        count("*").alias("registros_amostrais"),
        spark_sum("peso_amostral").alias("estimativa_ponderada")
    )
    .orderBy("id_tempo")
)

display(gold_indicadores_gerais)

In [0]:
# Persistência da tabela de indicadores gerais na Camada Gold

tabela_indicadores_gerais = "gold_indicadores_gerais"

(
    gold_indicadores_gerais
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_indicadores_gerais)
)

print("Tabela analítica criada com sucesso:")
print(tabela_indicadores_gerais)

### 4.2. Perfil por Sexo

A tabela analítica de perfil por sexo apresenta a distribuição dos entregadores plataformizados segundo as categorias de sexo registradas na PNAD Contínua.

Para cada período, são calculados o número de registros amostrais, a estimativa populacional ponderada e a participação percentual de cada categoria no total estimado de entregadores plataformizados.

Os percentuais são calculados a partir das estimativas ponderadas, e não diretamente da contagem de registros amostrais.

In [0]:
# Construção da tabela analítica de perfil por sexo

from pyspark.sql.functions import count, sum as spark_sum, col
from pyspark.sql.window import Window

gold_perfil_sexo = (
    spark.table("gold_fact_trabalhador")
    .filter(col("entregador_plataformizado") == 1)
    .join(
        spark.table("gold_dim_tempo"),
        on="id_tempo",
        how="left"
    )
    .join(
        spark.table("gold_dim_sexo"),
        on="id_sexo",
        how="left"
    )
    .groupBy(
        "id_tempo",
        "ano_pnad",
        "periodo_pnad",
        "id_sexo",
        "sexo_desc"
    )
    .agg(
        count("*").alias("registros_amostrais"),
        spark_sum("peso_amostral").alias("estimativa_ponderada")
    )
)

janela_ano = Window.partitionBy("id_tempo")

gold_perfil_sexo = (
    gold_perfil_sexo
    .withColumn(
        "percentual_ponderado",
        (
            col("estimativa_ponderada")
            / spark_sum("estimativa_ponderada").over(janela_ano)
            * 100
        )
    )
    .orderBy("id_tempo", "id_sexo")
)

display(gold_perfil_sexo)

In [0]:
# Persistência da tabela analítica de perfil por sexo

tabela_perfil_sexo = "gold_perfil_sexo"

(
    gold_perfil_sexo
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_perfil_sexo)
)

print("Tabela analítica criada com sucesso:")
print(tabela_perfil_sexo)

### 4.3. Perfil por Cor ou Raça

A tabela analítica de perfil por cor ou raça apresenta a distribuição dos entregadores plataformizados segundo as categorias registradas na PNAD Contínua.

Para cada período, são calculados o número de registros amostrais, a estimativa populacional ponderada e a participação percentual de cada categoria no total estimado de entregadores plataformizados.

Os percentuais são calculados a partir das estimativas ponderadas, preservando a distinção entre a composição da amostra e a estimativa da população.

In [0]:
# Construção da tabela analítica de perfil por cor ou raça

from pyspark.sql.functions import count, sum as spark_sum, col
from pyspark.sql.window import Window

gold_perfil_cor_raca = (
    spark.table("gold_fact_trabalhador")
    .filter(col("entregador_plataformizado") == 1)
    .join(
        spark.table("gold_dim_tempo"),
        on="id_tempo",
        how="left"
    )
    .join(
        spark.table("gold_dim_cor_raca"),
        on="id_cor_raca",
        how="left"
    )
    .groupBy(
        "id_tempo",
        "ano_pnad",
        "periodo_pnad",
        "id_cor_raca",
        "cor_raca_desc"
    )
    .agg(
        count("*").alias("registros_amostrais"),
        spark_sum("peso_amostral").alias("estimativa_ponderada")
    )
)

janela_ano_cor_raca = Window.partitionBy("id_tempo")

gold_perfil_cor_raca = (
    gold_perfil_cor_raca
    .withColumn(
        "percentual_ponderado",
        (
            col("estimativa_ponderada")
            / spark_sum("estimativa_ponderada").over(janela_ano_cor_raca)
            * 100
        )
    )
    .orderBy("id_tempo", "id_cor_raca")
)

display(gold_perfil_cor_raca)

In [0]:
# Persistência da tabela analítica de perfil por cor ou raça

tabela_perfil_cor_raca = "gold_perfil_cor_raca"

(
    gold_perfil_cor_raca
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_perfil_cor_raca)
)

print("Tabela analítica criada com sucesso:")
print(tabela_perfil_cor_raca)

### 4.4. Perfil por Faixa Etária

A tabela analítica de perfil por faixa etária apresenta a distribuição dos entregadores plataformizados segundo grupos etários construídos a partir da idade informada na PNAD Contínua.

Para cada período, são calculados o número de registros amostrais, a estimativa populacional ponderada e a participação percentual de cada faixa etária no total estimado de entregadores plataformizados.

A organização em faixas permite sintetizar a composição etária da população analisada e comparar descritivamente os dois períodos considerados no projeto.

In [0]:
# Construção da tabela analítica de perfil por faixa etária

from pyspark.sql.functions import count, sum as spark_sum, col
from pyspark.sql.window import Window

gold_perfil_faixa_etaria = (
    spark.table("gold_fact_trabalhador")
    .filter(col("entregador_plataformizado") == 1)
    .join(
        spark.table("gold_dim_tempo"),
        on="id_tempo",
        how="left"
    )
    .join(
        spark.table("gold_dim_faixa_etaria"),
        on="id_faixa_etaria",
        how="left"
    )
    .groupBy(
        "id_tempo",
        "ano_pnad",
        "periodo_pnad",
        "id_faixa_etaria",
        "faixa_etaria"
    )
    .agg(
        count("*").alias("registros_amostrais"),
        spark_sum("peso_amostral").alias("estimativa_ponderada")
    )
)

janela_ano_faixa = Window.partitionBy("id_tempo")

gold_perfil_faixa_etaria = (
    gold_perfil_faixa_etaria
    .withColumn(
        "percentual_ponderado",
        (
            col("estimativa_ponderada")
            / spark_sum("estimativa_ponderada").over(janela_ano_faixa)
            * 100
        )
    )
    .orderBy("id_tempo", "id_faixa_etaria")
)

display(gold_perfil_faixa_etaria)

In [0]:
# Persistência da tabela analítica de perfil por faixa etária

tabela_perfil_faixa_etaria = "gold_perfil_faixa_etaria"

(
    gold_perfil_faixa_etaria
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_perfil_faixa_etaria)
)

print("Tabela analítica criada com sucesso:")
print(tabela_perfil_faixa_etaria)

### 4.5. Perfil por Posição na Ocupação

A tabela analítica de posição na ocupação apresenta a distribuição dos entregadores plataformizados segundo a posição ocupacional registrada na PNAD Contínua.

Para cada período, são calculados o número de registros amostrais, a estimativa populacional ponderada e a participação percentual de cada categoria no total estimado de entregadores plataformizados.

A classificação é utilizada como uma característica ocupacional da população analisada. Sua interpretação não pressupõe, isoladamente, correspondência direta com as formas de autonomia, subordinação ou vínculo experimentadas pelos trabalhadores em suas relações com as plataformas.

In [0]:
# Construção da tabela analítica de perfil por posição na ocupação

from pyspark.sql.functions import count, sum as spark_sum, col
from pyspark.sql.window import Window

gold_perfil_posicao_ocupacao = (
    spark.table("gold_fact_trabalhador")
    .filter(col("entregador_plataformizado") == 1)
    .join(
        spark.table("gold_dim_tempo"),
        on="id_tempo",
        how="left"
    )
    .join(
        spark.table("gold_dim_posicao_ocupacao"),
        on="id_posicao_ocupacao",
        how="left"
    )
    .groupBy(
        "id_tempo",
        "ano_pnad",
        "periodo_pnad",
        "id_posicao_ocupacao",
        "posicao_ocupacao"
    )
    .agg(
        count("*").alias("registros_amostrais"),
        spark_sum("peso_amostral").alias("estimativa_ponderada")
    )
)

janela_ano_posicao = Window.partitionBy("id_tempo")

gold_perfil_posicao_ocupacao = (
    gold_perfil_posicao_ocupacao
    .withColumn(
        "percentual_ponderado",
        (
            col("estimativa_ponderada")
            / spark_sum("estimativa_ponderada").over(janela_ano_posicao)
            * 100
        )
    )
    .orderBy("id_tempo", "id_posicao_ocupacao")
)

display(gold_perfil_posicao_ocupacao)

In [0]:
# Persistência da tabela analítica de perfil por posição na ocupação

tabela_perfil_posicao_ocupacao = "gold_perfil_posicao_ocupacao"

(
    gold_perfil_posicao_ocupacao
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_perfil_posicao_ocupacao)
)

print("Tabela analítica criada com sucesso:")
print(tabela_perfil_posicao_ocupacao)

### 4.6. Atividade Principal

Antes da construção da tabela analítica de atividade principal, são examinados os códigos de atividade efetivamente presentes entre os entregadores plataformizados.

Essa etapa permite identificar a diversidade de atividades registradas na PNAD Contínua e avaliar a forma mais adequada de organização da informação para consumo analítico, evitando a criação de agrupamentos ou classificações que não estejam previamente fundamentados nos dados utilizados.

In [0]:
# Exploração das atividades principais entre os entregadores plataformizados

from pyspark.sql.functions import count, sum as spark_sum, col

exploracao_atividade_principal = (
    spark.table("gold_fact_trabalhador")
    .filter(col("entregador_plataformizado") == 1)
    .join(
        spark.table("gold_dim_tempo"),
        on="id_tempo",
        how="left"
    )
    .join(
        spark.table("gold_dim_atividade_principal"),
        on="id_atividade_principal",
        how="left"
    )
    .groupBy(
        "id_tempo",
        "ano_pnad",
        "periodo_pnad",
        "codigo_atividade_pnad"
    )
    .agg(
        count("*").alias("registros_amostrais"),
        spark_sum("peso_amostral").alias("estimativa_ponderada")
    )
    .orderBy(
        "id_tempo",
        col("estimativa_ponderada").desc()
    )
)

display(exploracao_atividade_principal)

A exploração dos dados identificou diferentes códigos de atividade principal entre os entregadores plataformizados, com concentração das estimativas em um conjunto mais restrito de atividades.

Nesta etapa, os códigos originais são preservados sem a criação de agrupamentos substantivos adicionais. Essa opção busca evitar a atribuição de categorias não previamente fundamentadas na documentação utilizada na construção da base.

A tabela analítica resultante permite examinar a distribuição dos entregadores entre os códigos de atividade registrados, mantendo aberta a possibilidade de enriquecimento posterior da dimensão a partir de documentação classificatória específica.

In [0]:
# Construção da tabela analítica de atividade principal

from pyspark.sql.functions import col, sum as spark_sum
from pyspark.sql.window import Window

janela_ano_atividade = Window.partitionBy("id_tempo")

gold_perfil_atividade_principal = (
    exploracao_atividade_principal
    .withColumn(
        "percentual_ponderado",
        (
            col("estimativa_ponderada")
            / spark_sum("estimativa_ponderada").over(janela_ano_atividade)
            * 100
        )
    )
    .orderBy(
        "id_tempo",
        col("estimativa_ponderada").desc()
    )
)

display(gold_perfil_atividade_principal)

In [0]:
# Persistência da tabela analítica de atividade principal

tabela_perfil_atividade_principal = "gold_perfil_atividade_principal"

(
    gold_perfil_atividade_principal
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_perfil_atividade_principal)
)

print("Tabela analítica criada com sucesso:")
print(tabela_perfil_atividade_principal)

## 5. Auditoria da Camada Gold

Após a construção do modelo dimensional e das tabelas analíticas, realiza-se uma verificação final da Camada Gold.

A auditoria busca confirmar a existência das tabelas persistidas e verificar se suas quantidades de registros permanecem compatíveis com as estruturas construídas e validadas ao longo do processo.

Essa etapa antecede a análise dos resultados e permite identificar eventuais perdas, duplicações ou inconsistências produzidas durante a persistência das tabelas.

In [0]:
# Auditoria das tabelas persistidas na Camada Gold

tabelas_gold = [
    "gold_dim_tempo",
    "gold_dim_sexo",
    "gold_dim_cor_raca",
    "gold_dim_faixa_etaria",
    "gold_dim_posicao_ocupacao",
    "gold_dim_atividade_principal",
    "gold_fact_trabalhador",
    "gold_indicadores_gerais",
    "gold_perfil_sexo",
    "gold_perfil_cor_raca",
    "gold_perfil_faixa_etaria",
    "gold_perfil_posicao_ocupacao",
    "gold_perfil_atividade_principal"
]

for tabela in tabelas_gold:
    quantidade = spark.table(tabela).count()
    print(f"{tabela}: {quantidade} registros")

### 5.1. Consistência das Estimativas entre as Tabelas Analíticas

Além da verificação estrutural das tabelas persistidas, é necessário avaliar a consistência dos indicadores produzidos.

Para isso, as estimativas ponderadas das diferentes tabelas analíticas são novamente agregadas por período e comparadas com os totais apresentados na tabela de indicadores gerais.

A correspondência entre esses valores permite verificar se os diferentes recortes analíticos preservam o mesmo universo estimado de entregadores plataformizados.

In [0]:
# Verificação da consistência das estimativas entre os marts Gold

tabelas_perfil = [
    "gold_perfil_sexo",
    "gold_perfil_cor_raca",
    "gold_perfil_faixa_etaria",
    "gold_perfil_posicao_ocupacao",
    "gold_perfil_atividade_principal"
]

for tabela in tabelas_perfil:
    
    print(f"\n{tabela}")
    
    (
        spark.table(tabela)
        .groupBy("ano_pnad")
        .agg(
            spark_sum("estimativa_ponderada")
            .alias("total_estimado")
        )
        .orderBy("ano_pnad")
        .show()
    )

## 6. Documentação do Modelo de Dados

A Camada Gold foi estruturada a partir de um modelo dimensional em esquema estrela, tendo a tabela `gold_fact_trabalhador` como núcleo do modelo.

A tabela fato preserva uma observação individual da PNAD Contínua por registro e concentra as medidas e indicadores necessários às análises, especialmente o peso amostral e os indicadores relacionados ao trabalho por plataformas.

Ao redor da tabela fato foram organizadas seis dimensões:

- `gold_dim_tempo`
- `gold_dim_sexo`
- `gold_dim_cor_raca`
- `gold_dim_faixa_etaria`
- `gold_dim_posicao_ocupacao`
- `gold_dim_atividade_principal`

As relações entre a tabela fato e as dimensões são estabelecidas por meio das respectivas chaves dimensionais.

Além do esquema dimensional, foram construídas tabelas analíticas agregadas para facilitar o consumo dos principais indicadores do projeto. Essas tabelas não alteram a granularidade ou a estrutura da tabela fato, funcionando como estruturas derivadas destinadas à análise e à visualização dos dados.

## 6.1 Diagrama Entidade-Relacionamento da Camada Gold

O modelo dimensional adotado organiza a `gold_fact_trabalhador` como tabela central, relacionada a seis dimensões: tempo, sexo, cor ou raça, faixa etária, posição na ocupação e atividade principal. Cada registro da tabela fato corresponde a uma observação individual da PNAD Contínua em determinado período, preservando o peso amostral necessário à produção das estimativas populacionais.

O campo `id_registro_fato` funciona como identificador técnico do registro na tabela fato e não corresponde a um identificador individual da pessoa pesquisada.

**Figura 1 — Modelo dimensional da Camada Gold**



In [0]:
import base64

caminho_der = "/Volumes/workspace/gold/artefatos_projeto/der_gold_entregadores.png"

with open(caminho_der, "rb") as arquivo:
    imagem_base64 = base64.b64encode(arquivo.read()).decode("utf-8")

displayHTML(
    f"""
    <div style="text-align:center;">
        <img src="data:image/png;base64,{imagem_base64}"
             style="max-width:100%; height:auto;">
        <p style="font-size:13px;">
            <em>Fonte: elaboração própria a partir da modelagem desenvolvida no projeto.</em>
        </p>
    </div>
    """
)

### Observação sobre o modelo

O DER representa o esquema estrela da Camada Gold e, portanto, contempla a tabela fato e as seis dimensões que compõem o modelo dimensional. As tabelas agregadas `gold_indicadores_gerais` e `gold_perfil_*` não são representadas no diagrama, pois constituem estruturas derivadas destinadas ao consumo analítico, e não dimensões adicionais do modelo.



## 7. Qualidade dos Dados

A qualidade dos dados foi avaliada ao longo das diferentes etapas do pipeline, desde a ingestão dos arquivos originais até a construção das estruturas analíticas da Camada Gold.

As verificações buscaram assegurar que as transformações realizadas não produzissem perdas ou duplicações indevidas de registros e que os indicadores utilizados nas análises permanecessem consistentes após a modelagem dimensional.

Foram considerados os seguintes aspectos:

- **Integridade da ingestão:** preservação da quantidade de registros provenientes dos arquivos originais da PNAD Contínua, com 478.091 observações em 2022 e 479.778 em 2024, totalizando 957.869 registros.

- **Consistência entre as camadas:** a integração e as transformações realizadas na Camada Silver preservaram o total de 957.869 registros provenientes da Bronze.

- **Validação da população analítica:** foram identificados 691 registros amostrais de entregadores plataformizados em 2022 e 778 em 2024, correspondentes a estimativas ponderadas de aproximadamente 445,9 mil e 487,3 mil trabalhadores, respectivamente.

- **Integridade da tabela fato:** a `gold_fact_trabalhador` preservou os 957.869 registros da Camada Silver, mantendo os controles relativos à população analítica e aos respectivos pesos amostrais.

- **Integridade dimensional:** foram verificadas as chaves utilizadas no relacionamento entre a tabela fato e as seis dimensões do modelo, não sendo identificadas chaves dimensionais nulas após a aplicação das categorias técnicas previstas na modelagem.

- **Consistência das agregações:** as tabelas analíticas de sexo, cor ou raça, faixa etária, posição na ocupação e atividade principal retornaram, quando agregadas por período, às mesmas estimativas populacionais apresentadas na tabela de indicadores gerais.

- **Persistência das estruturas:** as tabelas dimensionais, a tabela fato e as tabelas analíticas foram persistidas em formato Delta e posteriormente consultadas para verificar sua disponibilidade e consistência.

Essas verificações não eliminam as limitações próprias da fonte ou do desenho amostral da pesquisa. Seu objetivo é avaliar a consistência interna do pipeline desenvolvido e assegurar que os resultados analíticos possam ser rastreados até os dados tratados nas etapas anteriores.

# 8. Análise dos Dados e Solução do Problema

A construção do pipeline permite utilizar os microdados da PNAD Contínua para produzir uma caracterização quantitativa dos entregadores plataformizados nos períodos considerados no projeto.

As análises apresentadas nesta seção utilizam as estimativas ponderadas produzidas a partir da Camada Gold. A contagem dos registros amostrais é preservada como informação de controle, mas não deve ser confundida com a estimativa do número de trabalhadores na população.

A comparação entre 2022 e 2024 é realizada de forma descritiva, considerando os períodos específicos disponíveis na base utilizada. As diferenças observadas são tratadas como variações entre os dois recortes analisados, sem que isso implique, por si só, a identificação de tendências ou relações causais.

## 8.1. Estimativa de Entregadores Plataformizados

A primeira análise examina a estimativa ponderada de entregadores plataformizados em cada período, permitindo dimensionar a população identificada pelo critério analítico adotado no projeto.

In [0]:
# Gráfico 1 - Estimativa de entregadores plataformizados

import matplotlib.pyplot as plt

df_indicadores = (
    spark.table("gold_indicadores_gerais")
    .orderBy("ano_pnad")
    .toPandas()
)

plt.figure(figsize=(8, 5))

barras = plt.bar(
    df_indicadores["ano_pnad"].astype(str),
    df_indicadores["estimativa_ponderada"]
)

plt.title("Estimativa de entregadores plataformizados")
plt.xlabel("Período da PNAD Contínua")
plt.ylabel("Número estimado de trabalhadores")

rotulos_periodo = {
    2022: "4º trim. 2022",
    2024: "3º trim. 2024"
}

plt.xticks(
    range(len(df_indicadores)),
    [rotulos_periodo[ano] for ano in df_indicadores["ano_pnad"]]
)

for barra, valor in zip(barras, df_indicadores["estimativa_ponderada"]):
    plt.text(
        barra.get_x() + barra.get_width() / 2,
        barra.get_height(),
        f"{valor/1000:.1f} mil".replace(".", ","),
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

In [0]:
# Variação da estimativa de entregadores entre os períodos analisados

valor_2022 = df_indicadores.loc[
    df_indicadores["ano_pnad"] == 2022,
    "estimativa_ponderada"
].iloc[0]

valor_2024 = df_indicadores.loc[
    df_indicadores["ano_pnad"] == 2024,
    "estimativa_ponderada"
].iloc[0]

diferenca_absoluta = valor_2024 - valor_2022

variacao_percentual = (
    diferenca_absoluta / valor_2022
) * 100

print(f"Estimativa 2022: {valor_2022:,.0f}")
print(f"Estimativa 2024: {valor_2024:,.0f}")
print(f"Diferença absoluta: {diferenca_absoluta:,.0f}")
print(f"Variação percentual: {variacao_percentual:.2f}%")

### Interpretação

A estimativa ponderada indica aproximadamente **445,9 mil entregadores plataformizados no 4º trimestre de 2022** e **487,3 mil no 3º trimestre de 2024**. A diferença entre os dois períodos corresponde a cerca de **41,4 mil trabalhadores**, ou **9,3%** em relação à estimativa observada em 2022.

O resultado sugere uma presença numericamente mais elevada de entregadores plataformizados no período analisado em 2024. Essa diferença, contudo, deve ser interpretada com cautela. As duas observações correspondem a trimestres distintos da PNAD Contínua — quarto trimestre de 2022 e terceiro trimestre de 2024 — e, portanto, não constituem, isoladamente, uma série temporal capaz de demonstrar uma trajetória contínua de crescimento.

A comparação permite identificar uma diferença entre os dois períodos analisados, enquanto sua interpretação deve considerar os limites temporais e amostrais da fonte utilizada.

## 8.2. Composição por Sexo

A segunda análise examina a distribuição percentual dos entregadores plataformizados segundo as categorias de sexo registradas na PNAD Contínua.

Para permitir a comparação da composição entre os dois períodos, são utilizados os percentuais calculados a partir das estimativas ponderadas, e não diretamente das frequências observadas na amostra.

In [0]:
# Gráfico 2 - Composição dos entregadores plataformizados por sexo

import matplotlib.pyplot as plt
import numpy as np

df_sexo = (
    spark.table("gold_perfil_sexo")
    .orderBy("id_sexo", "ano_pnad")
    .toPandas()
)

df_sexo_pivot = (
    df_sexo
    .pivot(
        index="sexo_desc",
        columns="ano_pnad",
        values="percentual_ponderado"
    )
)

categorias = df_sexo_pivot.index
x = np.arange(len(categorias))
largura = 0.35

fig, ax = plt.subplots(figsize=(8, 5))

barras_2022 = ax.bar(
    x - largura/2,
    df_sexo_pivot[2022],
    largura,
    label="4º trim. 2022"
)

barras_2024 = ax.bar(
    x + largura/2,
    df_sexo_pivot[2024],
    largura,
    label="3º trim. 2024"
)

ax.set_title("Composição dos entregadores plataformizados por sexo")
ax.set_xlabel("Sexo")
ax.set_ylabel("Participação na estimativa de entregadores (%)")
ax.set_xticks(x)
ax.set_xticklabels(categorias)
ax.legend()

for barras in [barras_2022, barras_2024]:
    for barra in barras:
        valor = barra.get_height()
        ax.text(
            barra.get_x() + barra.get_width()/2,
            valor,
            f"{valor:.1f}%".replace(".", ","),
            ha="center",
            va="bottom"
        )

ax.set_ylim(0, 85)

plt.tight_layout()
plt.show()

### Interpretação

A composição por sexo apresenta pouca variação entre os dois períodos analisados. No 4º trimestre de 2022, os homens correspondiam a aproximadamente **76,0%** da estimativa de entregadores plataformizados, enquanto as mulheres representavam **24,0%**. No 3º trimestre de 2024, essas participações foram de **76,7%** e **23,3%**, respectivamente.

Os dados indicam, portanto, uma composição predominantemente masculina nos dois períodos, com proporções bastante próximas entre as duas observações. Mais do que sugerir uma mudança substantiva entre 2022 e 2024, a comparação evidencia a permanência de uma distribuição marcadamente desigual entre homens e mulheres no universo estimado de entregadores plataformizados.

Essa caracterização descreve a composição da população estimada nos períodos analisados e, isoladamente, não permite estabelecer as razões associadas à maior participação masculina na atividade.

## 8.3. Composição por Cor ou Raça

A terceira análise examina a distribuição dos entregadores plataformizados segundo as categorias de cor ou raça registradas na PNAD Contínua.

Assim como nas análises anteriores, são utilizados os percentuais derivados das estimativas ponderadas. A comparação busca caracterizar a composição observada em cada período e identificar diferenças descritivas entre os dois recortes analisados.

In [0]:
# Gráfico 3 - Composição dos entregadores plataformizados por cor ou raça

import matplotlib.pyplot as plt
import numpy as np

df_cor_raca = (
    spark.table("gold_perfil_cor_raca")
    .orderBy("id_cor_raca", "ano_pnad")
    .toPandas()
)

df_cor_raca_pivot = (
    df_cor_raca
    .pivot(
        index="cor_raca_desc",
        columns="ano_pnad",
        values="percentual_ponderado"
    )
)

# Mantém a ordem das categorias definida na dimensão
ordem_categorias = ["Branca", "Preta", "Amarela", "Parda", "Indígena"]

df_cor_raca_pivot = df_cor_raca_pivot.reindex(ordem_categorias)

categorias = df_cor_raca_pivot.index
x = np.arange(len(categorias))
largura = 0.35

fig, ax = plt.subplots(figsize=(10, 5))

barras_2022 = ax.bar(
    x - largura/2,
    df_cor_raca_pivot[2022],
    largura,
    label="4º trim. 2022"
)

barras_2024 = ax.bar(
    x + largura/2,
    df_cor_raca_pivot[2024],
    largura,
    label="3º trim. 2024"
)

ax.set_title("Composição dos entregadores plataformizados por cor ou raça")
ax.set_xlabel("Cor ou raça")
ax.set_ylabel("Participação na estimativa de entregadores (%)")
ax.set_xticks(x)
ax.set_xticklabels(categorias)
ax.legend()

for barras in [barras_2022, barras_2024]:
    for barra in barras:
        valor = barra.get_height()
        ax.text(
            barra.get_x() + barra.get_width()/2,
            valor,
            f"{valor:.1f}%".replace(".", ","),
            ha="center",
            va="bottom",
            fontsize=9
        )

ax.set_ylim(0, 52)

plt.tight_layout()
plt.show()

### Interpretação

A composição por cor ou raça apresenta maior concentração nas categorias branca e parda nos dois períodos analisados. No 4º trimestre de 2022, os entregadores classificados como pardos correspondiam a aproximadamente **44,2%** da estimativa, enquanto os brancos representavam **41,4%** e os pretos, **13,2%**. As categorias amarela e indígena apresentavam participações inferiores a 2%.

No 3º trimestre de 2024, a participação estimada dos brancos corresponde a **45,2%**, a dos pardos a **39,8%** e a dos pretos a **14,2%**. Em relação ao período observado em 2022, verifica-se, portanto, uma participação proporcionalmente maior da categoria branca e menor da categoria parda, enquanto a participação da categoria preta apresenta diferença mais reduzida.

Essas diferenças permitem caracterizar a composição estimada dos entregadores nos dois períodos, mas devem ser interpretadas de forma descritiva. A comparação entre duas observações da PNAD Contínua, realizadas em trimestres distintos, não permite atribuir às variações identificadas o sentido de uma transformação consolidada da composição racial dos entregadores plataformizados.

## 8.4. Composição por Faixa Etária

A quarta análise examina a composição etária dos entregadores plataformizados a partir das faixas construídas na Camada Gold.

Os percentuais são calculados sobre as estimativas ponderadas de cada período. A distribuição por faixas etárias permite identificar em quais grupos se concentra a população estimada de entregadores e comparar descritivamente sua composição entre os dois períodos analisados.

In [0]:
# Gráfico 4 - Composição dos entregadores plataformizados por faixa etária

import matplotlib.pyplot as plt
import numpy as np

df_faixa = (
    spark.table("gold_perfil_faixa_etaria")
    .orderBy("id_faixa_etaria", "ano_pnad")
    .toPandas()
)

df_faixa_pivot = (
    df_faixa
    .pivot(
        index="faixa_etaria",
        columns="ano_pnad",
        values="percentual_ponderado"
    )
)

ordem_faixas = [
    "14 a 17 anos",
    "18 a 24 anos",
    "25 a 34 anos",
    "35 a 44 anos",
    "45 a 54 anos",
    "55 a 64 anos",
    "65 anos ou mais"
]

df_faixa_pivot = df_faixa_pivot.reindex(ordem_faixas)

categorias = df_faixa_pivot.index
x = np.arange(len(categorias))
largura = 0.35

fig, ax = plt.subplots(figsize=(11, 5))

barras_2022 = ax.bar(
    x - largura/2,
    df_faixa_pivot[2022],
    largura,
    label="4º trim. 2022"
)

barras_2024 = ax.bar(
    x + largura/2,
    df_faixa_pivot[2024],
    largura,
    label="3º trim. 2024"
)

ax.set_title("Composição dos entregadores plataformizados por faixa etária")
ax.set_xlabel("Faixa etária")
ax.set_ylabel("Participação na estimativa de entregadores (%)")
ax.set_xticks(x)
ax.set_xticklabels(categorias)
ax.legend()

for barras in [barras_2022, barras_2024]:
    for barra in barras:
        valor = barra.get_height()
        ax.text(
            barra.get_x() + barra.get_width()/2,
            valor,
            f"{valor:.1f}%".replace(".", ","),
            ha="center",
            va="bottom",
            fontsize=8
        )

ax.set_ylim(0, 40)

plt.tight_layout()
plt.show()

In [0]:
# Participação conjunta dos entregadores entre 25 e 44 anos

faixas_centrais = [
    "25 a 34 anos",
    "35 a 44 anos"
]

participacao_25_44 = (
    df_faixa[
        df_faixa["faixa_etaria"].isin(faixas_centrais)
    ]
    .groupby("ano_pnad")["percentual_ponderado"]
    .sum()
)

for ano, percentual in participacao_25_44.items():
    print(
        f"{ano}: {percentual:.2f}% dos entregadores "
        "estão nas faixas de 25 a 44 anos"
    )

### Interpretação

A distribuição por faixa etária apresenta configuração semelhante nos dois períodos analisados, com maior concentração dos entregadores nas faixas intermediárias de idade. No 4º trimestre de 2022, a faixa de **25 a 34 anos** correspondia a aproximadamente **35,0%** da estimativa, seguida pela faixa de **35 a 44 anos**, com **27,0%**. No 3º trimestre de 2024, essas participações foram de **34,5%** e **26,6%**, respectivamente.

Consideradas conjuntamente, as faixas entre **25 e 44 anos** reuniam aproximadamente **62,0%** dos entregadores estimados em 2022 e **61,1%** em 2024. Os dados indicam, assim, que cerca de seis em cada dez entregadores plataformizados estavam concentrados nesse intervalo etário nos dois períodos observados.

As demais faixas apresentam participações menores e algumas diferenças entre os períodos, como o aumento da participação estimada dos trabalhadores entre 55 e 64 anos e a redução daquela correspondente aos trabalhadores com 65 anos ou mais. Diante da natureza descritiva da comparação, entretanto, essas diferenças não são tomadas isoladamente como evidência de uma mudança na estrutura etária da atividade.

## 8.5. Posição na Ocupação

A quinta análise examina a distribuição dos entregadores plataformizados segundo a posição na ocupação registrada na PNAD Contínua.

Os percentuais apresentados são calculados a partir das estimativas ponderadas de cada período. A classificação permite caracterizar a forma como esses trabalhadores aparecem situados nas categorias ocupacionais da pesquisa, sem pressupor correspondência direta entre essas categorias e os sentidos de autonomia ou subordinação presentes na experiência do trabalho por plataformas.

In [0]:
# Gráfico 5 - Composição dos entregadores por posição na ocupação

import matplotlib.pyplot as plt
import numpy as np

df_posicao = (
    spark.table("gold_perfil_posicao_ocupacao")
    .orderBy("id_posicao_ocupacao", "ano_pnad")
    .toPandas()
)

df_posicao_pivot = (
    df_posicao
    .pivot(
        index="posicao_ocupacao",
        columns="ano_pnad",
        values="percentual_ponderado"
    )
)

ordem_posicoes = [
    "Empregado do setor privado",
    "Empregador",
    "Conta própria",
    "Trabalhador familiar auxiliar"
]

df_posicao_pivot = df_posicao_pivot.reindex(ordem_posicoes)

categorias = df_posicao_pivot.index
x = np.arange(len(categorias))
largura = 0.35

fig, ax = plt.subplots(figsize=(11, 5))

barras_2022 = ax.bar(
    x - largura/2,
    df_posicao_pivot[2022],
    largura,
    label="4º trim. 2022"
)

barras_2024 = ax.bar(
    x + largura/2,
    df_posicao_pivot[2024],
    largura,
    label="3º trim. 2024"
)

ax.set_title("Composição dos entregadores plataformizados por posição na ocupação")
ax.set_xlabel("Posição na ocupação")
ax.set_ylabel("Participação na estimativa de entregadores (%)")
ax.set_xticks(x)
ax.set_xticklabels(categorias)
ax.legend()

for barras in [barras_2022, barras_2024]:
    for barra in barras:
        valor = barra.get_height()
        ax.text(
            barra.get_x() + barra.get_width()/2,
            valor,
            f"{valor:.1f}%".replace(".", ","),
            ha="center",
            va="bottom",
            fontsize=9
        )

ax.set_ylim(0, 82)

plt.tight_layout()
plt.show()

### Interpretação

A distribuição segundo a posição na ocupação apresenta configuração bastante semelhante nos dois períodos analisados. A categoria **conta própria** concentra aproximadamente três quartos da estimativa de entregadores plataformizados, correspondendo a **75,0%** no 4º trimestre de 2022 e **75,5%** no 3º trimestre de 2024.

A categoria **empregador** aparece em seguida, com participação estimada de **19,0%** em 2022 e **18,9%** em 2024. Os empregados do setor privado representam **5,2%** e **4,7%**, respectivamente, enquanto os trabalhadores familiares auxiliares permanecem abaixo de 1% nos dois períodos.

Mais do que indicar mudanças relevantes entre as duas observações, os resultados evidenciam a predominância da classificação como trabalhador por conta própria entre os entregadores plataformizados identificados pelo critério adotado no projeto. Essa classificação, entretanto, deve ser compreendida nos limites da categoria estatística utilizada pela PNAD Contínua. Sua predominância não permite concluir, isoladamente, que esses trabalhadores disponham de autonomia equivalente sobre as condições, os ritmos ou a organização de seu trabalho.

## 8.6. Atividade Principal

A Camada Gold também permite examinar os códigos de atividade principal associados aos entregadores plataformizados. A exploração identificou diferentes códigos de atividade nos dois períodos analisados, com concentração da estimativa em um conjunto mais restrito de códigos.

Nesta versão do projeto, optou-se por não produzir uma visualização categorial dessa variável. Embora os códigos originais tenham sido preservados na dimensão `gold_dim_atividade_principal` e na tabela analítica `gold_perfil_atividade_principal`, a documentação incorporada ao pipeline não contém, nesta etapa, a correspondência completa entre todos os códigos observados e suas respectivas denominações.

A manutenção dos códigos na Camada Gold preserva a informação disponível e permite seu enriquecimento posterior mediante incorporação do dicionário classificatório correspondente, sem necessidade de reconstrução das etapas anteriores do pipeline.

## 8.7. Entregadores Plataformizados e Demais Trabalhadores

Além da caracterização interna dos entregadores plataformizados, o modelo construído permite estabelecer comparações com os demais trabalhadores presentes no universo ocupacional considerado na base.

Essa comparação busca identificar em que medida algumas características observadas entre os entregadores se aproximam ou se diferenciam daquelas encontradas entre os demais trabalhadores. Para evitar a inclusão de registros fora do universo ocupacional, são considerados nesta análise os casos que apresentam posição na ocupação válida na tabela fato.

Os grupos são definidos de forma mutuamente exclusiva: entregadores plataformizados, identificados pelo indicador analítico construído no projeto, e demais trabalhadores pertencentes ao universo ocupacional considerado.

In [0]:
# Verificação do universo comparativo:
# entregadores plataformizados x demais trabalhadores

from pyspark.sql.functions import (
    col,
    when,
    count,
    sum as spark_sum
)

df_comparacao = (
    spark.table("gold_fact_trabalhador")
    .filter(col("id_posicao_ocupacao") != 99)
    .withColumn(
        "grupo_trabalhador",
        when(
            col("entregador_plataformizado") == 1,
            "Entregadores plataformizados"
        ).otherwise("Demais trabalhadores")
    )
    .join(
        spark.table("gold_dim_tempo"),
        on="id_tempo",
        how="left"
    )
)

controle_comparacao = (
    df_comparacao
    .groupBy(
        "ano_pnad",
        "periodo_pnad",
        "grupo_trabalhador"
    )
    .agg(
        count("*").alias("registros_amostrais"),
        spark_sum("peso_amostral").alias("estimativa_ponderada")
    )
    .orderBy("ano_pnad", "grupo_trabalhador")
)

display(controle_comparacao)

In [0]:
# Comparação da composição por sexo:
# entregadores plataformizados x demais trabalhadores

from pyspark.sql.functions import (
    col,
    count,
    sum as spark_sum
)
from pyspark.sql.window import Window

comparacao_sexo = (
    df_comparacao
    .join(
        spark.table("gold_dim_sexo"),
        on="id_sexo",
        how="left"
    )
    .groupBy(
        "ano_pnad",
        "periodo_pnad",
        "grupo_trabalhador",
        "id_sexo",
        "sexo_desc"
    )
    .agg(
        count("*").alias("registros_amostrais"),
        spark_sum("peso_amostral").alias("estimativa_ponderada")
    )
)

janela_grupo_ano = Window.partitionBy(
    "ano_pnad",
    "grupo_trabalhador"
)

comparacao_sexo = (
    comparacao_sexo
    .withColumn(
        "percentual_ponderado",
        (
            col("estimativa_ponderada")
            / spark_sum("estimativa_ponderada").over(janela_grupo_ano)
            * 100
        )
    )
    .orderBy(
        "ano_pnad",
        "grupo_trabalhador",
        "id_sexo"
    )
)

display(comparacao_sexo)

### 8.7.1. Comparação da Composição por Sexo

A comparação por sexo permite situar a composição dos entregadores plataformizados em relação aos demais trabalhadores pertencentes ao universo ocupacional considerado no projeto.

São comparadas as participações percentuais ponderadas de homens e mulheres em cada grupo e período, permitindo observar diferenças de composição sem que o tamanho absoluto das duas populações interfira diretamente na comparação.

In [0]:
# Gráfico 6 - Composição por sexo:
# entregadores plataformizados x demais trabalhadores

import matplotlib.pyplot as plt
import numpy as np

df_comp_sexo = comparacao_sexo.toPandas()

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

periodos = [
    (2022, "4º trim. 2022"),
    (2024, "3º trim. 2024")
]

for ax, (ano, titulo) in zip(axes, periodos):

    dados_ano = df_comp_sexo[
        df_comp_sexo["ano_pnad"] == ano
    ]

    pivot = dados_ano.pivot(
        index="sexo_desc",
        columns="grupo_trabalhador",
        values="percentual_ponderado"
    ).reindex(["Homem", "Mulher"])

    categorias = pivot.index
    x = np.arange(len(categorias))
    largura = 0.35

    barras_demais = ax.bar(
        x - largura/2,
        pivot["Demais trabalhadores"],
        largura,
        label="Demais trabalhadores"
    )

    barras_entregadores = ax.bar(
        x + largura/2,
        pivot["Entregadores plataformizados"],
        largura,
        label="Entregadores plataformizados"
    )

    ax.set_title(titulo)
    ax.set_xlabel("Sexo")
    ax.set_xticks(x)
    ax.set_xticklabels(categorias)
    ax.set_ylim(0, 85)

    for barras in [barras_demais, barras_entregadores]:
        for barra in barras:
            valor = barra.get_height()
            ax.text(
                barra.get_x() + barra.get_width()/2,
                valor,
                f"{valor:.1f}%".replace(".", ","),
                ha="center",
                va="bottom",
                fontsize=8
            )

axes[0].set_ylabel("Participação no respectivo grupo (%)")
axes[1].legend()

fig.suptitle(
    "Composição por sexo: entregadores plataformizados e demais trabalhadores"
)

plt.tight_layout()
plt.show()

### Interpretação

A comparação com os demais trabalhadores evidencia que a predominância masculina observada entre os entregadores plataformizados é mais acentuada do que aquela encontrada no universo ocupacional utilizado como referência.

No 4º trimestre de 2022, os homens correspondiam a aproximadamente **56,8%** dos demais trabalhadores e a **76,0%** dos entregadores plataformizados, uma diferença de cerca de **19,2 pontos percentuais**. No 3º trimestre de 2024, as respectivas participações foram de **56,5%** e **76,7%**, ampliando essa diferença para aproximadamente **20,2 pontos percentuais**.

Entre as mulheres, observa-se o movimento correspondente: sua participação entre os demais trabalhadores alcançava **43,2%** em 2022 e **43,5%** em 2024, enquanto entre os entregadores plataformizados correspondia a **24,0%** e **23,3%**, respectivamente.

A comparação sugere, portanto, que a composição dos entregadores plataformizados se distingue do conjunto dos demais trabalhadores considerado no projeto pela presença proporcionalmente mais elevada de homens. A relativa estabilidade desse contraste nos dois períodos reforça seu caráter descritivo, embora os dados utilizados não permitam, isoladamente, explicar os fatores associados a essa diferença.

### 8.7.2. Comparação da Composição por Faixa Etária

A comparação por faixa etária busca situar a composição etária dos entregadores plataformizados em relação aos demais trabalhadores pertencentes ao universo ocupacional considerado no projeto.

Para cada grupo e período, são calculadas as participações percentuais ponderadas das diferentes faixas etárias. Essa comparação permite observar se a concentração etária identificada entre os entregadores também se verifica entre os demais trabalhadores ou se assume configuração distinta.

In [0]:
# Comparação da composição por faixa etária:
# entregadores plataformizados x demais trabalhadores

from pyspark.sql.functions import col, count, sum as spark_sum
from pyspark.sql.window import Window

comparacao_faixa_etaria = (
    df_comparacao
    .join(
        spark.table("gold_dim_faixa_etaria"),
        on="id_faixa_etaria",
        how="left"
    )
    .groupBy(
        "ano_pnad",
        "periodo_pnad",
        "grupo_trabalhador",
        "id_faixa_etaria",
        "faixa_etaria"
    )
    .agg(
        count("*").alias("registros_amostrais"),
        spark_sum("peso_amostral").alias("estimativa_ponderada")
    )
)

janela_grupo_ano_faixa = Window.partitionBy(
    "ano_pnad",
    "grupo_trabalhador"
)

comparacao_faixa_etaria = (
    comparacao_faixa_etaria
    .withColumn(
        "percentual_ponderado",
        (
            col("estimativa_ponderada")
            / spark_sum("estimativa_ponderada").over(janela_grupo_ano_faixa)
            * 100
        )
    )
    .filter(col("id_faixa_etaria") != 99)
    .orderBy(
        "ano_pnad",
        "grupo_trabalhador",
        "id_faixa_etaria"
    )
)

display(comparacao_faixa_etaria)

In [0]:
# Gráfico 7 - Composição por faixa etária:
# entregadores plataformizados x demais trabalhadores

import matplotlib.pyplot as plt
import numpy as np

df_comp_faixa = comparacao_faixa_etaria.toPandas()

ordem_faixas = [
    "14 a 17 anos",
    "18 a 24 anos",
    "25 a 34 anos",
    "35 a 44 anos",
    "45 a 54 anos",
    "55 a 64 anos",
    "65 anos ou mais"
]

fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)

periodos = [
    (2022, "4º trim. 2022"),
    (2024, "3º trim. 2024")
]

for ax, (ano, titulo) in zip(axes, periodos):

    dados_ano = df_comp_faixa[
        df_comp_faixa["ano_pnad"] == ano
    ]

    pivot = (
        dados_ano
        .pivot(
            index="faixa_etaria",
            columns="grupo_trabalhador",
            values="percentual_ponderado"
        )
        .reindex(ordem_faixas)
    )

    categorias = pivot.index
    x = np.arange(len(categorias))
    largura = 0.35

    barras_demais = ax.bar(
        x - largura/2,
        pivot["Demais trabalhadores"],
        largura,
        label="Demais trabalhadores"
    )

    barras_entregadores = ax.bar(
        x + largura/2,
        pivot["Entregadores plataformizados"],
        largura,
        label="Entregadores plataformizados"
    )

    ax.set_title(titulo)
    ax.set_xlabel("Faixa etária")
    ax.set_xticks(x)
    ax.set_xticklabels(
        categorias,
        rotation=45,
        ha="right"
    )
    ax.set_ylim(0, 40)

    for barras in [barras_demais, barras_entregadores]:
        for barra in barras:
            valor = barra.get_height()

            ax.text(
                barra.get_x() + barra.get_width()/2,
                valor,
                f"{valor:.1f}%".replace(".", ","),
                ha="center",
                va="bottom",
                fontsize=7
            )

axes[0].set_ylabel("Participação no respectivo grupo (%)")
axes[1].legend()

fig.suptitle(
    "Composição por faixa etária: entregadores plataformizados e demais trabalhadores"
)

plt.tight_layout()
plt.show()

### Interpretação

A comparação por faixa etária evidencia diferenças na composição dos entregadores plataformizados em relação aos demais trabalhadores. Nos dois períodos analisados, a principal diferença aparece na faixa de **25 a 34 anos**, que corresponde a aproximadamente **35,0%** dos entregadores em 2022 e **34,5%** em 2024, frente a **25,0%** e **24,4%**, respectivamente, entre os demais trabalhadores.

Consideradas conjuntamente, as faixas entre **25 e 44 anos** concentram aproximadamente **62,0%** dos entregadores plataformizados em 2022 e **61,1%** em 2024. Entre os demais trabalhadores, essas mesmas faixas correspondem a cerca de **51,1%** e **52,1%**, respectivamente. A concentração nesse intervalo etário é, portanto, aproximadamente dez pontos percentuais maior entre os entregadores nos dois períodos observados.

Nas faixas a partir dos 45 anos, observa-se, em sentido inverso, participação proporcionalmente menor dos entregadores em comparação com os demais trabalhadores. O contraste é particularmente visível entre 55 e 64 anos e entre aqueles com 65 anos ou mais.

Os resultados sugerem uma composição etária específica entre os entregadores plataformizados no universo analisado, caracterizada menos por uma presença generalizada das faixas mais jovens do que por uma concentração relativa entre **25 e 44 anos**. A comparação permanece descritiva e não permite, isoladamente, explicar os fatores associados a essa configuração.

In [0]:
# Comparação da posição na ocupação:
# entregadores plataformizados x demais trabalhadores

from pyspark.sql.functions import col, count, sum as spark_sum
from pyspark.sql.window import Window

comparacao_posicao = (
    df_comparacao
    .join(
        spark.table("gold_dim_posicao_ocupacao"),
        on="id_posicao_ocupacao",
        how="left"
    )
    .groupBy(
        "ano_pnad",
        "periodo_pnad",
        "grupo_trabalhador",
        "id_posicao_ocupacao",
        "posicao_ocupacao"
    )
    .agg(
        count("*").alias("registros_amostrais"),
        spark_sum("peso_amostral").alias("estimativa_ponderada")
    )
)

janela_grupo_ano_posicao = Window.partitionBy(
    "ano_pnad",
    "grupo_trabalhador"
)

comparacao_posicao = (
    comparacao_posicao
    .withColumn(
        "percentual_ponderado",
        (
            col("estimativa_ponderada")
            / spark_sum("estimativa_ponderada").over(janela_grupo_ano_posicao)
            * 100
        )
    )
    .orderBy(
        "ano_pnad",
        "grupo_trabalhador",
        "id_posicao_ocupacao"
    )
)

display(comparacao_posicao)

### 8.7.3. Comparação da Posição na Ocupação

A comparação segundo a posição na ocupação permite situar a classificação ocupacional dos entregadores plataformizados em relação aos demais trabalhadores pertencentes ao universo considerado no projeto.

Para cada grupo e período, são comparadas as participações percentuais ponderadas das diferentes posições na ocupação. O objetivo é identificar diferenças na composição ocupacional, preservando a distinção entre a classificação estatística utilizada pela PNAD Contínua e as formas concretas de autonomia, subordinação ou vínculo experimentadas no trabalho.

In [0]:
# Gráfico 8 - Posição na ocupação:
# entregadores plataformizados x demais trabalhadores

import matplotlib.pyplot as plt
import numpy as np

df_comp_posicao = comparacao_posicao.toPandas()

ordem_posicoes = [
    "Trabalhador doméstico",
    "Militar",
    "Empregado do setor privado",
    "Empregado do setor público",
    "Empregador",
    "Conta própria",
    "Trabalhador familiar auxiliar"
]

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

periodos = [
    (2022, "4º trim. 2022"),
    (2024, "3º trim. 2024")
]

for ax, (ano, titulo) in zip(axes, periodos):

    dados_ano = df_comp_posicao[
        df_comp_posicao["ano_pnad"] == ano
    ]

    pivot = (
        dados_ano
        .pivot(
            index="posicao_ocupacao",
            columns="grupo_trabalhador",
            values="percentual_ponderado"
        )
        .reindex(ordem_posicoes)
        .fillna(0)
    )

    categorias = pivot.index
    x = np.arange(len(categorias))
    largura = 0.35

    barras_demais = ax.bar(
        x - largura/2,
        pivot["Demais trabalhadores"],
        largura,
        label="Demais trabalhadores"
    )

    barras_entregadores = ax.bar(
        x + largura/2,
        pivot["Entregadores plataformizados"],
        largura,
        label="Entregadores plataformizados"
    )

    ax.set_title(titulo)
    ax.set_xlabel("Posição na ocupação")
    ax.set_xticks(x)
    ax.set_xticklabels(
        categorias,
        rotation=45,
        ha="right"
    )
    ax.set_ylim(0, 82)

    for barras in [barras_demais, barras_entregadores]:
        for barra in barras:
            valor = barra.get_height()

            if valor > 0:
                ax.text(
                    barra.get_x() + barra.get_width()/2,
                    valor,
                    f"{valor:.1f}%".replace(".", ","),
                    ha="center",
                    va="bottom",
                    fontsize=7
                )

axes[0].set_ylabel("Participação no respectivo grupo (%)")
axes[1].legend()

fig.suptitle(
    "Posição na ocupação: entregadores plataformizados e demais trabalhadores"
)

plt.tight_layout()
plt.show()

### Interpretação

A comparação evidencia uma diferença expressiva na distribuição da posição na ocupação entre os entregadores plataformizados e os demais trabalhadores. Nos dois períodos analisados, aproximadamente três quartos dos entregadores aparecem classificados como trabalhadores por **conta própria**: **75,0%** no 4º trimestre de 2022 e **75,5%** no 3º trimestre de 2024. Entre os demais trabalhadores, essa categoria corresponde a **25,4%** e **24,4%**, respectivamente.

O contraste também aparece na categoria **empregado do setor privado**. Entre os demais trabalhadores, ela representa **50,6%** da estimativa em 2022 e **51,9%** em 2024, enquanto entre os entregadores plataformizados corresponde a apenas **5,2%** e **4,7%**. A distribuição observada apresenta, portanto, configurações ocupacionais distintas nos dois grupos e relativamente semelhantes entre os períodos analisados.

A predominância da categoria conta própria entre os entregadores constitui um resultado relevante para a caracterização quantitativa desse universo, sobretudo quando situada em relação aos demais trabalhadores. Essa classificação, contudo, não deve ser tomada como equivalente empírico de autonomia no trabalho. Ela informa a posição ocupacional pela qual esses trabalhadores são classificados na pesquisa, mas não permite, isoladamente, estabelecer o grau de controle que exercem sobre jornadas, remuneração, distribuição das atividades ou demais condições de realização do trabalho.

# 9. Síntese dos Resultados e Solução do Problema

O projeto teve como objetivo desenvolver um pipeline de dados capaz de transformar os microdados da PNAD Contínua em uma estrutura organizada e reutilizável para a análise quantitativa do trabalho de entregadores por aplicativo. A solução construída integra dados referentes ao 4º trimestre de 2022 e ao 3º trimestre de 2024 e organiza seu processamento segundo a arquitetura Medallion, com camadas Bronze, Silver e Gold.

Na Camada Bronze, os arquivos originais foram ingeridos e preservados em sua estrutura de origem. Na Camada Silver, foram realizadas a extração das variáveis selecionadas, a harmonização entre os dois períodos, o tratamento dos tipos de dados e a construção dos indicadores necessários à identificação da população analítica. Na Camada Gold, os dados foram organizados em um modelo dimensional composto por uma tabela fato e seis dimensões, complementado por tabelas agregadas destinadas ao consumo analítico.

A estrutura desenvolvida permitiu estimar aproximadamente **445,9 mil entregadores plataformizados no 4º trimestre de 2022** e **487,3 mil no 3º trimestre de 2024**. A diferença entre os dois períodos corresponde a aproximadamente **41,4 mil trabalhadores**, ou **9,3%** em relação à estimativa observada em 2022. Por se tratar de observações realizadas em trimestres distintos, essa diferença é interpretada de forma descritiva e não como demonstração isolada de uma trajetória contínua de crescimento.

A caracterização sociodemográfica mostrou relativa estabilidade na composição observada nos dois períodos. Os homens representam aproximadamente três quartos dos entregadores estimados, enquanto as faixas entre **25 e 44 anos** concentram cerca de seis em cada dez trabalhadores. A distribuição por cor ou raça apresenta maior participação das categorias branca e parda, embora suas proporções apresentem diferenças entre os dois períodos analisados.

A posição na ocupação constitui um dos resultados mais marcantes da análise. Aproximadamente **75% dos entregadores plataformizados aparecem classificados como trabalhadores por conta própria** nos dois períodos. Essa informação assume maior relevo quando comparada ao universo dos demais trabalhadores considerado no projeto, no qual a participação da categoria conta própria corresponde a aproximadamente um quarto da população estimada.

As análises comparativas também evidenciaram outras particularidades da composição dos entregadores. A participação masculina é aproximadamente vinte pontos percentuais superior àquela observada entre os demais trabalhadores, enquanto a concentração nas faixas entre 25 e 44 anos é cerca de dez pontos percentuais maior. Esses resultados permitem situar as características dos entregadores em relação a um universo ocupacional mais amplo, em vez de descrevê-las apenas de forma isolada.

A classificação predominante como trabalhador por conta própria não é interpretada, neste projeto, como evidência suficiente de autonomia substantiva no exercício da atividade. A variável informa uma posição ocupacional registrada pela pesquisa, enquanto questões relacionadas ao controle sobre jornadas, remuneração, distribuição das atividades e demais condições de trabalho demandam outras dimensões de investigação. Nesse sentido, os indicadores quantitativos produzidos podem ser colocados em diálogo com outras estratégias de pesquisa sem que uma abordagem seja utilizada como confirmação automática da outra.

Como produto, o pipeline permite que os microdados sejam processados de forma rastreável desde sua ingestão até a produção dos indicadores analíticos. A preservação de uma tabela fato com o universo harmonizado, associada às dimensões e às tabelas analíticas da Camada Gold, permite ampliar posteriormente as perguntas formuladas sem necessidade de reconstruir integralmente o processamento dos dados.

A solução desenvolvida constitui, assim, uma base inicial para um **Observatório de Dados dos Entregadores por Aplicativo**, capaz de organizar indicadores sobre dimensão estimada, composição sociodemográfica e características ocupacionais dessa população. Sua arquitetura permite incorporar posteriormente novos períodos, variáveis e fontes de dados, preservando a separação entre ingestão, tratamento, modelagem e consumo analítico.

# 10. Limitações e Possibilidades de Evolução

A solução desenvolvida constitui uma primeira versão de uma infraestrutura de dados voltada à análise dos entregadores por aplicativo. Seu desenho permite responder às perguntas definidas para o MVP, mas também apresenta limites relacionados à fonte utilizada, ao recorte temporal e ao conjunto de variáveis incorporado ao pipeline.

Uma primeira limitação refere-se aos períodos disponíveis para comparação. O projeto utiliza dados do 4º trimestre de 2022 e do 3º trimestre de 2024. Por corresponderem a trimestres distintos, as diferenças observadas entre as duas bases devem ser interpretadas como comparações entre períodos específicos, e não, isoladamente, como evidência de uma trajetória temporal contínua.

A segunda limitação decorre do escopo de variáveis selecionado para esta versão do pipeline. A Camada Silver foi estruturada para permitir a identificação dos entregadores plataformizados e sua caracterização segundo sexo, cor ou raça, idade, posição na ocupação e atividade principal. Outras dimensões potencialmente relevantes para a compreensão das condições de trabalho não foram incorporadas nesta etapa e poderão ser acrescentadas em versões posteriores da solução.

A atividade principal também apresenta uma limitação documental específica. Embora seus códigos tenham sido preservados na Camada Silver, modelados na dimensão correspondente e disponibilizados em uma tabela analítica da Camada Gold, sua utilização substantiva permanece condicionada à incorporação da correspondência classificatória completa entre os códigos observados e suas denominações. A arquitetura adotada permite realizar esse enriquecimento posteriormente sem reconstruir as demais etapas do pipeline.

Do ponto de vista analítico, os indicadores produzidos são predominantemente descritivos. As diferenças identificadas entre períodos ou entre grupos não são tomadas, por si mesmas, como evidência de relações causais. Da mesma forma, categorias estatísticas como posição na ocupação são tratadas nos limites de sua função classificatória, evitando sua equivalência automática com dimensões mais amplas da experiência do trabalho.

Esses limites também indicam possibilidades de desenvolvimento do projeto. A arquitetura Medallion e o modelo dimensional construído permitem incorporar novas variáveis, períodos e fontes, preservando a separação entre os dados originais, as transformações realizadas e as estruturas destinadas ao consumo analítico.

Entre as possibilidades de evolução está a ampliação dos indicadores relacionados às condições de trabalho, à jornada, aos rendimentos e a outras características disponíveis nas fontes que venham a ser incorporadas ao pipeline. A inclusão de novos períodos também poderá ampliar a capacidade de acompanhamento das mudanças observadas no trabalho por plataformas.

Outra possibilidade consiste no desenvolvimento de uma camada de visualização interativa sobre as tabelas Gold. Os marts produzidos já organizam os indicadores em estruturas adequadas ao consumo por ferramentas de visualização, permitindo que gráficos atualmente apresentados no notebook sejam posteriormente convertidos em painéis de acompanhamento do Observatório.

Por fim, a arquitetura pode ser ampliada para incorporar outras fontes de dados relacionadas ao trabalho por plataformas. Essa possibilidade permitiria construir um produto que não se limite à descrição de uma única base, mas organize diferentes conjuntos de informações em uma infraestrutura comum de análise.

Nesse sentido, o MVP não pretende esgotar as possibilidades de investigação quantitativa dos entregadores por aplicativo. Seu principal resultado é estabelecer uma estrutura inicial, reproduzível e extensível, a partir da qual o **Observatório de Dados dos Entregadores por Aplicativo** poderá ser progressivamente desenvolvido.

# 11. Autoavaliação

O desenvolvimento deste MVP permitiu transformar uma questão inicialmente vinculada à minha pesquisa sobre trabalho por plataformas em um problema de Engenharia de Dados. O principal desafio não consistiu apenas em produzir indicadores sobre os entregadores por aplicativo, mas em construir uma estrutura capaz de tornar esse processo organizado, rastreável e passível de expansão.

A utilização dos microdados da PNAD Contínua exigiu inicialmente compreender a estrutura dos arquivos, identificar as variáveis necessárias e compatibilizar informações provenientes de períodos distintos. Esse processo evidenciou a importância da documentação das fontes e da validação sucessiva das transformações realizadas, sobretudo diante de arquivos de largura fixa e de alterações na posição das variáveis entre as bases utilizadas.

A arquitetura Medallion contribuiu para organizar essas etapas. A Camada Bronze preservou os dados em sua forma original; a Silver concentrou as operações de extração, harmonização e construção dos indicadores; e a Gold reorganizou as informações para consumo analítico. Essa separação tornou mais clara a função de cada etapa e permitiu identificar com maior precisão onde cada transformação deveria ocorrer.

A construção da Camada Gold também representou um aprendizado importante sobre modelagem dimensional. Em vez de produzir diretamente tabelas e gráficos a partir dos microdados tratados, foi estruturado um esquema estrela composto por uma tabela fato e seis dimensões. A preservação do universo harmonizado na tabela fato, em lugar da manutenção exclusiva dos registros de entregadores, mostrou-se particularmente relevante ao permitir posteriormente a construção de análises comparativas com os demais trabalhadores.

Outro aspecto central foi a utilização do peso amostral. Ao longo do projeto, tornou-se necessário distinguir continuamente a quantidade de registros existentes na amostra das estimativas populacionais produzidas a partir desses registros. Essa distinção orientou tanto a construção das tabelas analíticas quanto a interpretação dos resultados apresentados.

As verificações de qualidade realizadas ao longo do pipeline também foram fundamentais. A preservação dos totais entre as camadas, a validação das chaves dimensionais, a conferência das estimativas ponderadas e a comparação dos totais produzidos pelos diferentes marts permitiram verificar a consistência interna da solução antes da etapa de análise.

Do ponto de vista dos resultados, o pipeline possibilitou não apenas caracterizar os entregadores plataformizados segundo diferentes dimensões sociodemográficas e ocupacionais, mas também situar algumas dessas características em relação aos demais trabalhadores. Essa comparação mostrou-se especialmente relevante nas análises de sexo, faixa etária e posição na ocupação, ampliando as possibilidades analíticas inicialmente previstas para o projeto.

Algumas decisões também envolveram reconhecer os limites da versão desenvolvida. A opção por preservar os códigos de atividade principal sem atribuir denominações não documentadas, por exemplo, restringiu uma das visualizações possíveis, mas manteve a rastreabilidade dos dados e evitou introduzir classificações que não haviam sido adequadamente validadas no pipeline.

Considero que o MVP atingiu seu objetivo ao construir uma primeira infraestrutura de dados para o **Observatório de Dados dos Entregadores por Aplicativo**, integrando ingestão, transformação, modelagem, validação e análise em um fluxo reproduzível. Ao mesmo tempo, sua construção tornou mais claros os caminhos para continuidade do produto, especialmente por meio da incorporação de novas variáveis, períodos, fontes e formas de visualização.

Como aprendizado, o desenvolvimento do projeto mostrou que a Engenharia de Dados não constitui apenas uma etapa técnica anterior à análise. As escolhas relativas à granularidade, às regras de transformação, à modelagem e aos controles de qualidade interferem diretamente naquilo que posteriormente pode ser perguntado aos dados e nos limites das interpretações produzidas a partir deles.